# Repetitions

> ### Learning Objectives
>
> By the end of this chapter you should be able to work with:
>
> - The purpose of repetition (loops) and when looping is preferable to sequential code
> - `while` loops as condition-controlled loops, including the fact that the body may run zero times
> - Sentinel-controlled `while` loops for validating input and reasoning about post-loop guarantees
> - `for` loops and the `range()` function with `start`, `stop`, and `step` (and why `stop` is exclusive)
> - Nested loops and predicting the number of iterations and resulting output
> - Infinite loops, their common causes, and how to avoid them
> - Applying loops to simulation and problem solving, including Monte Carlo estimation of π

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SETUP — run this cell first.
#
#  It draws every figure used in this chapter and switches the notebook into
#  "show me every result" mode.  Everything it needs is right here: nothing to
#  install, nothing to download, no other files required.
#
#  (Curious what a figure is made of?  The drawing code is all below.)
# ══════════════════════════════════════════════════════════════════════
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Circle
from matplotlib.lines import Line2D
from IPython.display import Image, display
from IPython.core.interactiveshell import InteractiveShell

# echo the value of *every* expression in a cell, the way the Python prompt
# does — many examples in this book show several results at once
InteractiveShell.ast_node_interactivity = "all"

# ------------------------------------------------------------- drawing ---

INK = "#1a1a1a"

MUTED = "#6b7280"

FILL = "#eef2f7"

ACCENT = "#2563eb"

WARM = "#b45309"

EDGE = "#334155"

def _frame(ax, xlim, ylim, title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color=MUTED, pad=8)

def _box(ax, xy, text, w=2.6, h=0.9, fc=FILL, ec=EDGE, fs=9, bold=False):
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK,
            zorder=3, fontweight="bold" if bold else "normal")
    return xy

def _diamond(ax, xy, text, w=3.0, h=1.5, fc="#fff7ed", ec=WARM, fs=9):
    x, y = xy
    ax.add_patch(Polygon(
        [(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)],
        closed=True, linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK, zorder=3)
    return xy

def _dot(ax, xy, r=0.09):
    ax.add_patch(Circle(xy, r, facecolor=EDGE, edgecolor=EDGE, zorder=4))
    return xy

def _arrow(ax, pts, label=None, label_at=0.5, label_off=(0.0, 0.18),
           color=EDGE, ha="center"):
    """Poly-line arrow through `pts` (elbow routing), head on the last segment."""
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    ax.add_line(Line2D(xs[:-1] + [xs[-1]], ys[:-1] + [ys[-1]],
                       color=color, linewidth=1.3, zorder=1,
                       solid_capstyle="round"))
    ax.annotate("", xy=pts[-1], xytext=pts[-2],
                arrowprops=dict(arrowstyle="-|>", color=color, linewidth=1.3,
                                shrinkA=0, shrinkB=0), zorder=1)
    if label:
        i = max(0, min(len(pts) - 2, int(label_at * (len(pts) - 1))))
        mx = (pts[i][0] + pts[i + 1][0]) / 2 + label_off[0]
        my = (pts[i][1] + pts[i + 1][1]) / 2 + label_off[1]
        ax.text(mx, my, label, fontsize=8, color=MUTED, ha=ha, va="center")

def _line(ax, pts, color=EDGE):
    """Poly-line with no arrowhead — for merging branches into a shared rail."""
    ax.add_line(Line2D([p[0] for p in pts], [p[1] for p in pts], color=color,
                       linewidth=1.3, zorder=1, solid_capstyle="round"))

def _cellgrid(ax, values, origin=(0, 0), cw=1.0, ch=1.0, fs=13, fc="white"):
    """A row/table of boxed cells; `values` is a list of rows."""
    x0, y0 = origin
    for r, row in enumerate(values):
        for c, v in enumerate(row):
            x = x0 + c * cw
            y = y0 - r * ch
            ax.add_patch(plt.Rectangle((x, y - ch), cw, ch, facecolor=fc,
                                       edgecolor=EDGE, linewidth=1.2, zorder=2))
            ax.text(x + cw / 2, y - ch / 2, str(v), ha="center", va="center",
                    fontsize=fs, color=INK, family="monospace", zorder=3)

def _mockwindow(ax, w, h, title, body, titlebar="#d7dde5", face="#ffffff",
                fs=9, textcolor=INK):
    """A framed window with a title bar and monospaced body lines."""
    ax.add_patch(plt.Rectangle((0, 0), w, h, facecolor=face, edgecolor=EDGE,
                               linewidth=1.2, zorder=1))
    ax.add_patch(plt.Rectangle((0, h - 0.55), w, 0.55, facecolor=titlebar,
                               edgecolor=EDGE, linewidth=1.2, zorder=2))
    ax.text(0.2, h - 0.28, title, fontsize=9, va="center", color=INK, zorder=3)
    y = h - 1.05
    for line, colour in body:
        ax.text(0.25, y, line, fontsize=fs, va="center", family="monospace",
                color=colour or textcolor, zorder=3)
        y -= 0.5

def _index_grid(ax, items, top_label, side_label, fs=13):
    n = len(items)
    _cellgrid(ax, [items], origin=(0, 1), fs=fs)
    for i in range(n):
        ax.text(i + 0.5, 1.25, str(i), ha="center", va="bottom", fontsize=10,
                color=ACCENT)
    if top_label:
        ax.text(-0.25, 1.3, top_label, ha="right", va="bottom", fontsize=9,
                color=ACCENT)
    if side_label:
        ax.text(-0.25, 0.5, side_label, ha="right", va="center", fontsize=9,
                color=ACCENT)
    _frame(ax, (-6.4, n + 0.4), (-0.4, 2.0))

def _double_diamond(ax, stage=None):
    names = ["Understand", "Design", "Implement", "Evaluate"]
    # two diamonds: centres at x=2.6 and x=7.8, half-width 2.6, half-height 2.0
    for d, cx in enumerate((2.6, 7.8)):
        left, right, top, bot = cx - 2.6, cx + 2.6, 2.0, -2.0
        for half in (0, 1):
            name = names[2 * d + half]
            tri = ([(left, 0), (cx, top), (cx, bot)] if half == 0
                   else [(cx, top), (right, 0), (cx, bot)])
            on = (name == stage)
            ax.add_patch(Polygon(tri, closed=True, zorder=1,
                                 facecolor="#b9bfc7" if on else "#eceef1",
                                 edgecolor="none"))
            tx = cx - 1.3 if half == 0 else cx + 1.3
            ax.text(tx, 0, name, ha="center", va="center", fontsize=10,
                    color=INK if on else MUTED,
                    fontweight="bold" if on else "normal", zorder=3)
        ax.add_patch(Polygon([(left, 0), (cx, top), (right, 0), (cx, bot)],
                             closed=True, facecolor="none", edgecolor=INK,
                             linewidth=2.2, zorder=2))
        ax.plot([cx, cx], [top, bot], color=INK, linewidth=1.0, zorder=2)
    for x, y in ((0.0, 0.0), (5.2, 0.0), (10.4, 0.0)):
        ax.add_patch(Circle((x, y), 0.22, facecolor="#c9ced6", edgecolor=INK,
                            linewidth=1.2, zorder=4))
    ax.plot([-1.5, -0.22], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([10.62, 11.9], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([5.2, 5.2], [-0.22, -3.1], color=INK, linewidth=1.6, zorder=2)
    ax.text(-1.7, 0, "Problem", ha="right", va="center", fontsize=10)
    ax.text(12.1, 0, "Program", ha="left", va="center", fontsize=10)
    ax.text(5.2, -3.35, "Specification", ha="center", va="top", fontsize=10)
    _frame(ax, (-4.2, 14.2), (-4.2, 2.6))

def draw_flow_while(ax):
    """Replaces the TikZ flowchart in 06_Loops (while-loop)."""
    _box(ax, (0, 4.4), "code before the while-loop", w=4.8)
    join = _dot(ax, (0, 3.2))
    _diamond(ax, (0, 1.9), "while condition:", w=3.4)
    _box(ax, (4.6, 1.9), "execute block", w=2.8)
    _box(ax, (0, 0.0), "code after the while-loop", w=4.6)
    _arrow(ax, [(0, 3.95), (0, 3.2)])
    _arrow(ax, [join, (0, 2.65)])
    _arrow(ax, [(1.7, 1.9), (3.2, 1.9)], "True", label_off=(0, .22))
    _arrow(ax, [(0, 1.15), (0, 0.45)], "False", label_off=(-0.42, 0), ha="right")
    _arrow(ax, [(4.6, 2.35), (4.6, 3.2), (0.05, 3.2)])
    _frame(ax, (-3.0, 6.6), (-0.7, 5.1))

# ---------------------------------------------------------------- runtime ---
_SIZES = {'flow_while': (6.6, 3.8)}
_WIDTHS = {}
_FIGURES = {}


def _render(name):
    fig, ax = plt.subplots(figsize=_SIZES.get(name, (6.4, 4.4)), dpi=110)
    globals()["draw_" + name](ax)
    fig.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def show(name, width=None):
    """Display one of this chapter's figures."""
    display(Image(_FIGURES[name], width=width or _WIDTHS.get(name, 560)))


for _n in ['flow_while']:
    _FIGURES[_n] = _render(_n)

print("Setup complete \u2014 1 figure(s) ready.")


### Learning Objectives
 * identify and correctly author Python language syntax for repetition:  while loops and for loops;
* trace by hand the flow of program execution for programs that use while-loops and for-loops;
* design and author Python programs that use one or more loops; and
* describe what is an infinite loop.

Very frequently in computer programming we would like to repeat certain actions.  Sometimes we want to repeat these actions a specific number of times. Other times, we want to repeat some actions as long as some specified condition (i.e. Boolean expression) is `True`.  Sometimes we'd like to repeat some actions for every element of data in some collection of data elements.  In Python, we can do all of these things using *loops*.

### While-Loops

While-loops work a lot like an if-statement in that they have very similar syntax --- a condition followed by a block --- but the block can be executed multiple times as long as the condition is `True`.  While-loops consist of the word `while`, followed by a Boolean expression (the *loop condition*), followed by a colon, followed by a block of code.  Below you can see the general form of a while-loop, and the corresponding flow of execution presented as a flowchart.

"`

while condition:
	# block (indented)

In [ ]:
show("flow_while")

When execution of code reaches a while-loop, the loop's condition is evaluated.  The condition must be a Boolean expression yielding a result of `True` or `False`.  If the condition is `True`, the block of code following the while-loop's condition is repeated until the condition becomes `False`.  Then the (unindented) code after the while-loop executes.  Note that it is possible that the loop condition is `False` the first time it is encountered.  If this is the case, then the block is never executed, and execution proceeds to the code after the while-loop.

A while-loop can help us improve our guessing game from Section .  Previously, we asked the user to input a number between 1 and 100, and reported whether the guess was too high, too low, or valid.  But we had no easy mechanism to ask the user for a new guess if their guess was too high or too low.  With while-loops, we can repeat the actions of asking for a guess, and checking it for validity until the user enters a guess that is valid!

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
guess = int(input("Guess a number between 1 and 100: "))
while guess < 1 or guess > 100:
	if guess < 1:
		# If guess was less than one, execute this block.
		print("Too low!")
	elif guess > 100:
		# Otherwise, if guess is larger than 100, do this block.
		print("Too high!")
		
	# ask for a new guess
	guess = int(input("Guess a number between 1 and 100: "))

print("That was a valid guess!")

This program will ask the user for a guess, and then, as long as the guess is not valid, the while-loop's condition will be `True`, the reason for the guess being invalid will be printed, the user will be asked for another guess, then the loop condition will be checked again with the new guess, and so on, until the guess becomes valid.  Once the guess is valid, the loop condition will be `False`, and the `print` function call after the while-loop's block will print that it was a valid guess.

Note that the block after the while-loop's condition consists of the if-elif statement, and the line that asks for another guess.  The if-elif statement, in turn has its own blocks, which are indented relative to the first block.  This is an example of *nested blocks*.  You can nest blocks to any number of levels so long as all of the blocks at the same level are indented by exactly the same amount throughout the entire program.

If we are to run our new guessing program, the output will be as follows (green text is text entered by the user):

Output:
```text
Guess a number between 1 and 100: 125
Too high!
Guess a number between 1 and 100: 0
Too low!
Guess a number between 1 and 100: 42
That was a valid guess!
```

We will show you more examples of while-loops in class.

### While Loops for Counting

While loops can also be used to execute a block of code a pre-determined number of times.  These are called *counting loops* because an integer variable is used to count the number of times the block has executed, and the loop condition is such that the condition is `True` as long as the loop has executed fewer than the required number of times.  For example, we can use a counting while-loop to print a sentence a given number of times:

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
sentence = input("Enter a sentence: ")
n = int(input("How many times do you want to print it? "))

count = 0
while count < n:
    print(sentence)
    count = count + 1

In this example we start by asking the user what sentence we should print and how many times we want to print the sentence. The variable `count` acts as a counter that keeps track of how many times we've already printed it. To achieve our purpose we set the while-loop's condition to `count < times`, this causes the while-loop's block to execute until we have printed exactly `n` times the sentence.  The last line of the block where `count` is increased by 1 is very important for this to work.  If you leave this line out, you will get what is known as an *infinite loop* (see Section ).

The general form of a counting while-loop that does something $n$ times

In [ ]:
n = 10          # number of times you want to do something
counter = 0
while counter < n:
    print("doing the thing, repetition", counter)
    counter = counter + 1

Of course counting while-loops are not limited to those that count from 0 to $n$.  You can count from any integer $a$ to any other larger integer $b$ in a similar manner by changing the initialization of the counter variable so it starts counting at $a$, and adjusting the loop condition so the loop stops repeating when the counter's value is $b$.

### For-Loops

In Python, for-loops allow repetition of a block of code for each data item in a sequence (we will learn more about *sequences* in Section ). Right now we know about one kind of sequence: strings.  So we can use a for-loop to do something for every character in a string.  In this example, we have a function that counts and returns the number of capital letters in a string:

In [ ]:
def countCaps(s):
    count = 0
    for character in s:
        if character.isupper():
            count = count + 1
    return count

The block following the for-loop (consisting of the if-statement and its block) is executed once for each character in the string `s`; each time the block is repeated, the variable `character` refers to the next character in the string.

In general, the syntax of a for-loop consists of the word `for`, followed by a variable name, followed by the word `in`, followed by a sequence, followed by a colon, followed by a block:

```python
for variable in sequence:
	# Block of code -- each time this block is repeated,
	# variable refers to the next item in the sequence.
	# Repetition stops after each item in the sequence has 
	# been processed.
```

When we do something for each element of a sequence we say that we are *iterating* over the sequence. For-loops can be used to iterate over any sequence, not just strings.  In the next section we will introduce another kind of sequence called a *range* which is a sequence of integers.   We will learn about even more types of sequences in later chapters.

### Ranges and Counting For-Loops

We can use for-loops to create counting loops just like we did with while-loops.  To do so, we first need to learn about a new kind of sequence called a *range*.

A *range* is a sequence of integers that begins at an integer $a$ (the *start*), ends **before** an integer $b$ (the *stop*), and in which the difference between each element in the sequence, called the *step size*, is equal.  Ranges are created with Python's built-in `range` function.  The `range` function requires two arguments, *start* and *stop*, and can optionally accept a third argument for the step size which, if not given, defaults to 1.  You may also provide just a single argument to range; `range(x)` is equivalent to `range(0, x, 1)`, and is the sequence $0, 1, 2, \ldots, x-1$.
Here are some example ranges:

> *A fragment for illustration — it is not complete enough to run.*
```python
range(0,5,1)     # the sequence 0, 1, 2, 3, 4
range(5)         # the sequence 0, 1, 2, 3, 4
range(-4, 4)     # the sequence -4, -3, -2, -1, 0, 1, 2, 3
range(0, 11, 2)  # the sequence 0, 2, 4, 6, 8, 10
range(2, -3, -1) # the sequence 2, 1, 0, -1, -2
range(0, 5, 10)  # the sequence 0

## General form:
range(start, stop, step_size)
```

Remember:  the value `{stop}` is **not** part of the sequence.

Ranges can be used to write counting for-loops.  Here is a for-loop that repeats its block exactly `N` times:

In [ ]:
N = 5
for i in range(N):
    print(i)     # do something

In this loop, `i` refers to the value 0 on the first repetition, 1 on the second repetition, and so on, up to `N-1` on the last repetition.  It is equivalent to the following while-loop:

In [ ]:
N = 5
i = 0
while i < N:
    print(i)     # do something
    i = i + 1

### Nested Loops

In Section , we saw how to use nested branches. Similarly, complex iterations may require a loop inside another loop, we call this a *nested loop*. In this example we use a nested loop to print a multiplication table from 1 to 10. The first (outer) loop iterates over all rows of the
table. The second (inner) loop prints the columns in the current row.

In [ ]:
for i in range(1, 11):
    print()
    for j in range(1, 11):
        print(f"{i*j:4}", end="")

The `print` statement in this example looks a little bit different from what we have seen so far. First you can see that it receives a second argument `end=""`, this indicates that the print statement should not end with a new line. Secondly, the `f` in front of the string literal indicates that the string will be formatted. As a result any expression inside braces will be evaluated and formatted. In this case the value of the arithmetic expression `i*j` will be calculated and 4 spaces will be used for the answer (that is indicated by `:4`). You can learn more about formatting strings here: <http://docs.python.org/3/tutorial/inputoutput.html>.

The output of this code looks like this (try out running the program without the formatting and see how it changes):

```

   1   2   3   4   5   6   7   8   9  10
   2   4   6   8  10  12  14  16  18  20
   3   6   9  12  15  18  21  24  27  30
   4   8  12  16  20  24  28  32  36  40
   5  10  15  20  25  30  35  40  45  50
   6  12  18  24  30  36  42  48  54  60
   7  14  21  28  35  42  49  56  63  70
   8  16  24  32  40  48  56  64  72  80
   9  18  27  36  45  54  63  72  81  90
  10  20  30  40  50  60  70  80  90 100

```

### Choosing the Right Kind of Loop

Generally, for-loops are what you want to use to iterate over a sequence.  Both for-loops and while-loops are appropriate for simple counting loops.  You may prefer using for-loops with ranges for counting purposes because it requires less typing than the equivalent while-loop.  For most other non-counting loops that have complicated loop conditions and/or don't involve iterating over sequences, while-loops are likely the best choice.

### Sentinel Loops

There are a number of situations where we want a program to continue to run indefinitely until a particular condition is reached.  For example, when a traffic light turns on, it will continue to rotate between stop, yield and go until it is either shut off, or if there is an anomalous state reached (e.g. flashing yield).  Another example is in a video game such as Space Invaders --- the game to continues moving the invaders right, then down, then left, until all of the space invaders have been defeated or the player has been defeated.

In these situations, and many others, programmers will take advantage of a technique called a *sentinel loop*.  A sentinel loop is one that watches for a particular value or condition that will make it stop.  One of the most common uses for this particular technique is tracking user input when you do not know how many pieces of information a user will need to put in.  For example, in the following program calculates average rainfall based on user input, and will keep accepting numbers until the user enters a negative number.

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
#User inputs x weeks of rainfall to get
#an average over x weeks.
total_rainfall = 0
week_counter = 0
#Enter the first value outside of the loop
# so the loop has a value to compare.
rainfall =  int(input("Enter the amount of rainfall " + \
            "for each week, use -1 to stop:"))

 #loop until the sentinel value (-1) is entered
while rainfall != -1:
    total_rainfall = total_rainfall + rainfall
    week_counter = week_counter + 1
    rainfall =  int(input("Enter the amount of rainfall " + \
                "for each week, use -1 to stop:"))

#print the average rainfall
print ("The average rainfall over", week_counter, \
       "weeks is",total_rainfall/week_counter)

Note that the input statement for rainfall has to be repeated twice in this particular example because we need to have the rainfall variable in place for the loop to work on its first pass. It is possible to avoid this by including an initialization statement for the variable `rainfall` before the loop.

### Infinite Loops

Infinite loops are loops that repeat forever; they are usually the result of a bug in the code.  A while-loop whose loop condition can never become `False` is an infinite loop.  Here are a couple of examples:

> ⚠️ **This code is deliberately incorrect** — it is shown to illustrate a mistake, not to be run.
```python
## This counting loop is infinite because the programmer forgot
## to add the x = x + 1 line to the end of the block.  The value
## of x never changes, so the loop condition is always True.
x = 0
total = 0
while x < 10
	total = total + x
average = total / 10
```

> ⚠️ **This loop never ends.** It is shown to illustrate a bug; do not run it.
```python
## This loop is infinite because the programmer incorrectly used 
## 'or' instead of 'and'.  Mathematically, the condition can 
## never be False, regardless of the value referred to by x.
## Thus, the loop repeats forever.
x = -1
while x >= 0 or x <= 10:
	x = input("Enter a number that isn't between 0 and 10:")
```

It's quite difficult to accidentally write infinite for-loops because sequences are of finite length and they repeat only once for each item in the sequence.